In [ ]:
!pip install langchain langchain-openai langgraph python-dotenv langchain-community duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. 

### Set up Groq API Key

To use the Groq API, you'll need an API key. If you don't already have one, create one on the [Groq Console](https://console.groq.com/keys).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GROQ_API_KEY`.

In [19]:
# Used to securely store your API key
from google.colab import userdata

# Retrieve the Groq API key from Colab secrets
GROQ_API_KEY = "GROQ_KEY"

# Ensure the .env file exists and write the key to it
with open('.env', 'a') as f:
    f.write(f'GROQ_API_KEY="{GROQ_API_KEY}"\n')

print("GROQ_API_KEY written to .env file.")

GROQ_API_KEY written to .env file.


In [20]:
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq


load_dotenv()
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)

# --- shared state schema ---
class AgentState(TypedDict):
    goal:       str
    tasks:      List[str]
    results:    List[str]
    critique:   str
    approved:   bool
    iterations: int

In [10]:
def planner(state: AgentState) -> AgentState:
    system = """You are a planning agent. Break the user's goal into
at most 5 concrete, actionable tasks. Respond ONLY with a
valid JSON array of strings. No preamble, no markdown."""

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]
    response = llm.invoke(messages).content.strip()

    try:
        clean = response.replace("```json","").replace("```","").strip()
        tasks = json.loads(clean)
    except json.JSONDecodeError:
        tasks = [response]   # fallback: treat whole response as one task

    print(f"\n[Planner] Generated {len(tasks)} tasks:")
    for i, t in enumerate(tasks): print(f"  {i+1}. {t}")

    return {**state, "tasks": tasks}

In [13]:
#! pip install langchain langchain-openai langgraph python-dotenv langchain-community duckduckgo-search
# Imports and Environment (OPEN_AI key setup)
!pip install langchain_groq
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END

/tmp/ipykernel_1482/1683130421.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [1]:
%%capture
!pip install langchain langchain_groq langchain_community python-dotenv

In [15]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 988.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 22.8 MB/s eta 0:00:00


In [16]:
search = DuckDuckGoSearchRun()
def executor(state:AgentState) -> AgentState:
  results = []
  critique_ctx = ""
  if state["critique"]:
    critique_ctx = f"\n\nYour previous attempt was rejected. Previous critique: {state['critique']}"

  for task in state["tasks"]:
    system=f"""You are an execution agent. Complete the task thoroughly. Use web search if you need
    current information. {critique_ctx}"""

    # try web search for research tasks
    search_ctx = ""

    try:
      search_result = search.run(task[:100])
      search_ctx = f"\n\nWeb search result for context: \n{search_result[:800]}"
    except:
      pass

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Task: {task}{search_ctx}")
    ]

    result = llm.invoke(messages).content

    results.append(result)
    print(f"\n[Executor] Task: {task[:60]}...\n  Result: {result}")
  return {**state, "results": results, "iterations": state["iterations"] + 1}

In [17]:
# Agent with LLM-as-a-judge
def verifier(state: AgentState) -> AgentState:
    # safety net — approve after 3 iterations regardless
    if state["iterations"] >= 3:
        print("[Verifier] Max iterations reached — force approving.")
        return {**state, "approved": True}

    combined_results = "\n\n".join(
        f"Task {i+1}: {t}\nResult: {r}"
        for i, (t, r) in enumerate(zip(state["tasks"], state["results"]))
    )
    system = """You are a quality verifier. Evaluate the results against the
original goal using this rubric:
- Completeness: Does it fully address the goal? (0-0.4)
- Accuracy:     Is the information correct and specific? (0-0.3)
- Clarity:      Is it well-structured and clear? (0-0.3)
- Latency:      Does it take a reasonable amount of time to complete? (0-0.3)
- Tokens:       Number of tokens used (0-10000)
Sum the scores for a total between 0.0 and 1.0.
Respond ONLY as JSON: {"score":0.9, "completeness_score": 0.35, "accuracy_score": 0.2, "clarity_score":0.15, "approved": true, "critique": "...", latenc}"""

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Original goal: {state['goal']}\n\nResults:\n{combined_results}")
    ]
    raw = llm.invoke(messages).content.strip()
    try:
        clean = raw.replace("```json","").replace("```","").strip()
        verdict = json.loads(clean)
        completeness  = verdict.get("completeness_score", False)
        accuracy      = verdict.get("accuracy_score", False)
        clarity       = verdict.get("clarity_score", False)
        approved      = verdict.get("approved", False)
        critique  = verdict.get("critique", "")
        score     = verdict.get("score", 0)
    except:
        approved, critique, score = False, raw, 0

    print(f"\n[Verifier] Score: {score:.2f} | Approved: {approved}")
    if not approved: print(f"  Critique: {critique}")
    return {**state, "approved": approved, "critique": critique}

def route_after_verify(state: AgentState) -> str:
    return "end" if state["approved"] else "executor"

In [21]:
# --- run it ---
######Main######
graph = StateGraph(AgentState)

graph.add_node("planner",  planner)
graph.add_node("executor", executor)
graph.add_node("verifier", verifier)


graph.add_edge("planner", "executor")
graph.add_edge("executor", "verifier")
graph.add_conditional_edges(
    "verifier",
    route_after_verify,
    {"end": END, "executor": "executor"}
)

graph.set_entry_point("planner")


app = graph.compile()
initial_state: AgentState = {
    "goal":       "Research and summarise the top 3 trends in agriculture for 2025",
    "tasks":      [],
    "results":    [],
    "critique":   "",
    "approved":   False,
    "iterations": 0
}

final_state = app.invoke(initial_state)


[Planner] Generated 5 tasks:
  1. Identify credible sources of agricultural research and trends
  2. Research and gather information on the top 3 trends in agriculture for 2025
  3. Analyze and categorize the gathered information into relevant trends
  4. Evaluate and prioritize the top 3 trends based on relevance and impact
  5. Create a concise summary of the top 3 trends in agriculture for 2025

[Executor] Task: Identify credible sources of agricultural research and trend...
  Result: **Credible Sources of Agricultural Research and Trends:**

To identify credible sources of agricultural research and trends, I've compiled a list of reputable journals, organizations, and databases that provide reliable and up-to-date information on the latest developments in the field.

**Journals:**

1. **Journal of Agricultural Education and Extension**: A leading international journal that publishes research on agricultural education, extension, and communication.
2. **Agricultural Systems**: A jo